# Laterality score for HESTA and MOSTA sagittal sections

**Goal.** Estimate how far each sagittal section lies from the body midline,
so that HESTA (human) and MOSTA (mouse) sections at matched developmental
stages can be paired by comparable mediolateral position.

Neither dataset stores a section coordinate along the left–right axis, so the
position is inferred from *what the section cuts through*:

1. **Composition depth `D`** (0 = midline, 1 = lateral). Every annotated organ
   label is mapped to one of four depth levels:
   - *axial* (0): structures that only exist on the midline – spinal cord,
     notochord, cloaca, pituitary, …
   - *juxta-axial* (1/3): structures directly beside the spinal cord – dorsal
     root ganglia, somites, myotome/dermomyotome, sympathetic chain
   - *paramedian* (2/3): paired organs further out – kidneys (meso/meta-nephros),
     gonads, adrenals, lungs, …
   - *lateral* (1): structures far from the midline – limb buds, eye, inner ear, …
   - *ignored*: labels present at every depth (cavity, blood, brain, liver,
     skin, mesenchyme, …).

   `D` is the area-weighted mean depth over all non-ignored spots. Splitting the
   former single "paramedian" class in two separates sections that cut the
   ganglion/somite chain next to the cord from sections that only cut kidney and
   lung (in the first full run, ten MOSTA sections scored `L = 0.74–0.78`).
2. **Axial coverage `C`** (0 – 1). The geodesic length of the largest connected
   piece of axial tissue divided by the geodesic length of the section, capped
   at 1 (curled embryos can short-cut from head to tail; `C_raw` keeps the
   uncapped value).
3. **Combined laterality `L = w_D·D + (1−w_D)·(1−C)`** in [0, 1].

**Section order (HESTA only).** HESTA section numbers follow the cutting order,
i.e. the mediolateral position, but the cutting direction (left→right or
right→left) differs between embryos. Along a series the *distance* to the
midline must therefore be valley-shaped in the section number: decreasing,
then increasing (a monotone series is the special case of a valley at one end).
For every HESTA embryo we fit the best valley-shaped profile to `L`
(two isotonic regressions around the best split). This gives
- a smoothed score `L_fit`,
- a **relative side**: sections before the valley lie on one side of the
  midline (side `A`), sections after it on the other (side `B`), the valley
  section(s) are marked `0`. Which of `A`/`B` is the left side is unknown,
  because the cutting direction is unknown;
- a **validation statistic**: a joint permutation test. Its statistic is the
  total valley misfit (sum of squared residuals) over all embryos; the null
  distribution comes from random section orders within every embryo.

A V-shape is only accepted if it fits clearly better than a monotone trend
(`MIN_VALLEY_GAIN`); otherwise the series is treated as monotone and all its
sections are put on the same side, so that small wiggles are not mistaken for
a midline crossing.

MOSTA section numbers do not follow the position (sections 1 and 2 seem to be
the representative ones), so no order constraint is used there.

**Left vs. right.** `L` cannot tell the sides apart (the mirror symmetry from
Problem 2), and MOSTA sides are unknown. The pairing table therefore flags
every pair that needs to be aligned in **both orientations** (one section
reflected), unless both sections are at the midline. HESTA sections with the
same relative side in one embryo must receive the same orientation. The
exploratory PITX2/Pitx2 side hint is still reported, but the first full run
showed that it tracks stage, not side.

**Caveats.**
- Scores are comparable *within an embryo* more reliably than *across stages*,
  because body proportions and annotation granularity differ.
- The notebook prints every label that is not in the depth dictionary – extend
  `DEPTH_CLASS` when new files are added.
- Only the label column, `obsm['spatial']`, the gene names and (optionally) one
  gene column are read, directly with `h5py` (chunked scan for the gene), so the
  multi-GB files are never loaded into memory. anndata's backed mode is avoided
  because it still loads every layer (e.g. `counts`) into memory.

In [ ]:
import sys
sys.path.append("..")

import re
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, sparse
from scipy.sparse import csgraph
from scipy.spatial import cKDTree

import anndata as ad
from mgw.config import get_data_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Configuration

In [ ]:
# Folders holding the .h5ad files. None: the hesta/ and mouse_embryo/ subfolders of the
# MGW data directory (from get_data_dir(), as in demos/hesta_align.ipynb).
HESTA_DIR = None   # e.g. "/path/to/hesta_downloads"
MOSTA_DIR = None   # e.g. "/path/to/mosta_downloads"

if HESTA_DIR is None:
    HESTA_DIR = get_data_dir() / "hesta"  # asks on first run, then remembered per machine
if MOSTA_DIR is None:
    MOSTA_DIR = get_data_dir() / "mouse_embryo"
HESTA_DIR = Path(HESTA_DIR).expanduser()
MOSTA_DIR = Path(MOSTA_DIR).expanduser()
print("HESTA data directory:", HESTA_DIR)
print("MOSTA data directory:", MOSTA_DIR)

OUT_DIR = Path("laterality")  # results are written next to this notebook
OUT_DIR.mkdir(exist_ok=True)

DATASETS = {
    # name: (folder, glob pattern, obs column with organ labels, filename regex)
    "HESTA": (HESTA_DIR, "*_HESTA.h5ad", "celltype",
              r"^(?P<stage>CS[\d\-–]+)_E(?P<embryo>\d+)S(?P<section>\d+)"),
    "MOSTA": (MOSTA_DIR, "*.MOSTA.h5ad", "annotation",
              r"^(?P<stage>E[\d.]+)_E(?P<embryo>\d+)S(?P<section>\d+)"),
}
for ds, (folder, pattern, *_) in DATASETS.items():
    files = sorted(folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f"no {pattern} files in {folder}. Set {ds}_DIR above, or call "
                                "get_data_dir(reset=True) to change the data directory.")
    print(f"{ds}: {len(files)} sections: {', '.join(f.name for f in files)}")

# Gene used for the left/right side hint, and the organs in which it is evaluated.
SIDE_GENE = {"HESTA": "PITX2", "MOSTA": "Pitx2"}
SIDE_ORGANS = ["heart", "gut", "gi tract", "stomach"]
COMPUTE_SIDE_HINT = True  # scans X once per file (chunked); set False to skip

W_COMPOSITION = 0.5  # weight w_D of the composition depth in the combined score

# Stage correspondence (HESTA sample -> candidate MOSTA days), from the stage table.
STAGE_PAIRS = {
    "CS12-13": ["E9.5"],
    "CS14-15": ["E10.5"],
    "CS17": ["E11.5", "E12.5"],
    "CS18": ["E12.5"],
    "CS19": ["E12.5", "E13.5"],
    "CS20": ["E13.5"],
    "CS23": ["E14.5"],
}

# Section order. HESTA numbers follow the cutting order (direction unknown);
# MOSTA numbers do not, so only HESTA is fitted with a valley-shaped profile.
ORDERED_DATASETS = {"HESTA"}
MIN_SECTIONS_FOR_FIT = 3     # embryos with fewer sections are not fitted
MAX_FIT_RMS = 0.08           # larger residuals: fit flagged, raw L used for pairing
N_PERMUTATIONS = 20_000      # Monte Carlo permutations for series longer than 8
EXACT_PERMUTATION_MAX_N = 8  # up to this length all permutations are enumerated
MIN_VALLEY_GAIN = 0.01      # a V must reduce the SSE by this much vs. a monotone trend,
                             # otherwise the series is treated as monotone (one side only)
MIDLINE_TOL = 0.10           # L below this: section treated as midline (no mirror test)
RNG = np.random.default_rng(0)


### Organ label -> depth class

Keys are lower-case label names (both datasets). Classes: `axial`, `juxta_axial`,
`paramedian`, `lateral`, `ignore`. Unknown labels are reported below and treated
as `ignore`.

In [ ]:
DEPTH_VALUE = {"axial": 0.0, "juxta_axial": 1 / 3, "paramedian": 2 / 3, "lateral": 1.0}

DEPTH_CLASS = {
    # --- axial (0): exist only on / at the midline
    "spinal cord": "axial",
    "notochord": "axial",
    "cloaca": "axial",
    "urethra": "axial",
    "pituitary": "axial",
    "pituitary gland": "axial",
    "tongue": "axial",
    "thyroid": "axial",
    "esophagus": "axial",
    "trachea": "axial",
    "bladder": "axial",
    # --- juxta-axial (1/3): directly beside the spinal cord
    "ganglion": "juxta_axial",
    "dorsal root ganglion": "juxta_axial",
    "sympathetic nerve": "juxta_axial",
    "somite": "juxta_axial",
    "myotome": "juxta_axial",
    "dermomyotome": "juxta_axial",
    # --- paramedian (2/3): paired organs further from the midline
    "mesonephron": "paramedian",
    "mesonephros": "paramedian",
    "pronephron": "paramedian",
    "kidney": "paramedian",
    "metanephros": "paramedian",
    "urogenital ridge": "paramedian",
    "gonad": "paramedian",
    "testis": "paramedian",
    "ovary": "paramedian",
    "adrenal gland": "paramedian",
    "lung": "paramedian",
    "lung primordium": "paramedian",
    "thymus": "paramedian",
    "olfactory epithelium": "paramedian",
    # --- lateral (1): far from the midline
    "limb bud": "lateral",
    "limb": "lateral",
    "limb ectoderm": "lateral",
    "eye": "lateral",
    "lens": "lateral",
    "retina": "lateral",
    "inner ear": "lateral",
    "otic vesicle": "lateral",
    "submandibular gland": "lateral",
    "spleen": "lateral",
    # --- ignore: present at every depth / not informative
    "cavity": "ignore",
    "extraembryonic tissue": "ignore",
    "erythroblasts": "ignore",
    "blood vessel": "ignore",
    "endothelium": "ignore",
    "agm": "ignore",
    "brain": "ignore",
    "meninges": "ignore",
    "choroid plexus": "ignore",
    "heart": "ignore",
    "liver": "ignore",
    "hepatic diverticulum": "ignore",
    "gut": "ignore",
    "gi tract": "ignore",
    "primitive gut": "ignore",
    "primitive gut tube": "ignore",
    "stomach": "ignore",
    "pancreas": "ignore",
    "pancreas bud": "ignore",
    "pancreas primordium": "ignore",
    "mesentery": "ignore",
    "mesothelium": "ignore",
    "epidermis": "ignore",
    "surface ectoderm": "ignore",
    "connective tissue": "ignore",
    "mesenchyme": "ignore",
    "head mesenchyme": "ignore",
    "neural crest": "ignore",
    "sclerotome": "ignore",
    "cartilage": "ignore",
    "cartilage primordium": "ignore",
    "bone": "ignore",
    "cranial skeleton": "ignore",
    "skeletal muscle": "ignore",
    "muscle": "ignore",
    "smooth muscle": "ignore",
    "adipose tissue": "ignore",
    "branchial arch": "ignore",
    "pharyngeal": "ignore",
    "jaw and tooth": "ignore",
    "mucosal epithelium": "ignore",
    "facial nerve": "ignore",
}

# Labels that are not tissue of the embryo (excluded from the tissue mask).
NON_TISSUE = {"cavity", "extraembryonic tissue"}
AXIAL_LABELS = {k for k, v in DEPTH_CLASS.items() if v == "axial"}

## Loading (obs labels + coordinates only)

In [ ]:
def _strings(ds):
    return np.asarray(ds.asstr()[:], dtype=str)


def read_section(path, label_col):
    """Return (DataFrame with x, y, label; gene names) for one .h5ad file.

    Reads only these three elements with h5py. anndata's backed mode would also
    load every layer (e.g. counts) into memory, and AnnData objects are freed only
    by the cyclic garbage collector, so several of them can pile up across files.
    Handles the current categorical encoding and the older on-disk layout of the
    MOSTA files (codes referencing obs/__categories).
    """
    with h5py.File(path, "r") as h:
        col = h["obs"][label_col]
        if isinstance(col, h5py.Group):  # categorical: codes + categories
            codes, cats = col["codes"][:], _strings(col["categories"])
        elif "categories" in col.attrs:  # old layout
            codes, cats = col[:], _strings(h[col.attrs["categories"]])
        else:  # plain strings
            codes, cats = None, None
        labels = _strings(col) if codes is None else np.where(codes >= 0, cats[codes], "nan")
        xy = np.asarray(h["obsm"]["spatial"][:, :2], dtype=float)
        genes = _strings(h["var"][h["var"].attrs["_index"]])
    return pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "label": labels}), genes


def gene_vector(path, gene, genes, layer=None):
    """Expression of one gene for all spots, read from a CSR matrix in chunks."""
    hit = np.flatnonzero(genes == gene)
    if hit.size == 0:
        return None
    col = hit[0]
    with h5py.File(path, "r") as h:
        M = h["X"] if layer is None else h["layers"][layer]
        fmt = M.attrs.get("encoding-type", M.attrs.get("h5sparse_format", ""))
        if fmt not in ("csr_matrix", "csr"):
            raise ValueError(f"expected a CSR matrix, got {fmt!r}")
        indptr = M["indptr"][:]
        n_obs = len(indptr) - 1
        out = np.zeros(n_obs, dtype=np.float32)
        step = 2000
        for r0 in range(0, n_obs, step):
            r1 = min(r0 + step, n_obs)
            a, b = indptr[r0], indptr[r1]
            idx = M["indices"][a:b]
            sel = np.flatnonzero(idx == col)
            if sel.size:
                dat = M["data"][a:b][sel]
                rows = np.searchsorted(indptr[r0:r1 + 1], a + sel, side="right") - 1 + r0
                out[rows] = dat
    return out


def parse_name(path, regex):
    m = re.match(regex, Path(path).name)
    if m is None:
        return {"stage": "?", "embryo": "?", "section": "?"}
    d = m.groupdict()
    d["stage"] = d["stage"].replace("–", "-")
    return d

## Features

In [ ]:
MAX_RASTER_PIXELS = 50_000_000  # guard against a wrongly estimated grid step


def rasterize(df):
    """Map spot coordinates to integer grid indices (one pixel per bin).

    Bin spacing and grid orientation are estimated from nearest-neighbour vectors,
    so sections whose grid is rotated (e.g. MOSTA E11.5_E1S2, rotated by 10°) are
    turned back onto the axes instead of producing a huge raster.
    """
    xy = df[["x", "y"]].to_numpy(float)
    dist, nn = cKDTree(xy).query(xy, k=2)
    step = np.median(dist[:, 1])
    v = xy[nn[:, 1]] - xy
    theta = np.angle(np.mean(np.exp(4j * np.arctan2(v[:, 1], v[:, 0])))) / 4  # modulo 90°
    c, s = np.cos(theta), np.sin(theta)
    xy = xy @ np.array([[c, -s], [s, c]])  # rotate by -theta
    ij = np.round((xy - xy.min(axis=0)) / step).astype(int)
    shape = tuple(ij.max(axis=0) + 1)
    if shape[0] * shape[1] > MAX_RASTER_PIXELS:
        raise ValueError(f"raster {shape} too large (step {step:g}, angle {np.degrees(theta):.1f}°)")
    return ij[:, 0], ij[:, 1], shape


def largest_component(mask):
    lab, n = ndimage.label(mask, structure=np.ones((3, 3)))
    if n == 0:
        return np.zeros_like(mask)
    sizes = ndimage.sum(mask, lab, index=np.arange(1, n + 1))
    return lab == (np.argmax(sizes) + 1)


def geodesic_length(mask):
    """Longest shortest path inside a binary mask (double BFS), in pixels."""
    pts = np.argwhere(mask)
    if len(pts) < 2:
        return 0.0
    index = -np.ones(mask.shape, dtype=np.int64)
    index[tuple(pts.T)] = np.arange(len(pts))
    rows, cols, w = [], [], []
    for dx, dy in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        q = pts + [dx, dy]
        ok = (q[:, 0] >= 0) & (q[:, 0] < mask.shape[0]) & (q[:, 1] >= 0) & (q[:, 1] < mask.shape[1])
        j = np.full(len(pts), -1)
        j[ok] = index[q[ok, 0], q[ok, 1]]
        good = j >= 0
        rows.append(np.flatnonzero(good)); cols.append(j[good])
        w.append(np.full(good.sum(), np.hypot(dx, dy)))
    rows, cols, w = map(np.concatenate, (rows, cols, w))
    G = sparse.coo_matrix((w, (rows, cols)), shape=(len(pts), len(pts))).tocsr()
    d0 = csgraph.dijkstra(G, directed=False, indices=0)
    far = int(np.nanargmax(np.where(np.isfinite(d0), d0, -1)))
    d1 = csgraph.dijkstra(G, directed=False, indices=far)
    return float(np.max(d1[np.isfinite(d1)]))


def section_features(df):
    lab = df.label.str.lower()
    cls = lab.map(DEPTH_CLASS).fillna("ignore")
    counts = cls.value_counts()
    n_cls = sum(counts.get(c, 0) for c in DEPTH_VALUE)
    D = (sum(counts.get(c, 0) * v for c, v in DEPTH_VALUE.items()) / n_cls) if n_cls else np.nan

    ix, iy, shape = rasterize(df)
    tissue = np.zeros(shape, bool)
    t = ~lab.isin(NON_TISSUE).values
    tissue[ix[t], iy[t]] = True
    tissue = largest_component(ndimage.binary_fill_holes(ndimage.binary_closing(tissue, iterations=2)))
    axial = np.zeros(shape, bool)
    a = lab.isin(AXIAL_LABELS).values
    axial[ix[a], iy[a]] = True
    axial = largest_component(ndimage.binary_closing(axial, iterations=2))
    L_body, L_axial = geodesic_length(tissue), geodesic_length(axial)
    # In curled (C-shaped) embryos the longest path through the tissue can short-cut
    # between head and tail, so a full-length spinal cord can exceed it: cap at 1.
    C_raw = L_axial / L_body if L_body > 0 else np.nan
    C = min(1.0, C_raw)

    n = len(df)
    return {
        "n_spots": n,
        **{f"frac_{c}": counts.get(c, 0) / n for c in DEPTH_VALUE},
        "frac_classified": n_cls / n,
        "D_composition": D,
        "C_axial_coverage": C,
        "C_raw": C_raw,
        "body_length_px": L_body,
        "axial_length_px": L_axial,
        "L_laterality": W_COMPOSITION * D + (1 - W_COMPOSITION) * (1 - C),
    }


def side_hint(df, expr):
    """Fraction of heart/gut spots expressing the left-side gene (exploratory)."""
    if expr is None:
        return np.nan, 0
    m = df.label.str.lower().isin(SIDE_ORGANS).values
    return (float((expr[m] > 0).mean()) if m.sum() else np.nan), int(m.sum())

## Run on all available sections

In [ ]:
sections, rows, unknown = {}, [], set()
for ds, (folder, pattern, label_col, regex) in DATASETS.items():
    for path in sorted(folder.glob(pattern)):
        meta = parse_name(path, regex)
        df, genes = read_section(path, label_col)
        unknown |= {(ds, l) for l in df.label.str.lower().unique() if l not in DEPTH_CLASS}
        feats = section_features(df)
        if COMPUTE_SIDE_HINT:
            frac, n_side = side_hint(df, gene_vector(path, SIDE_GENE[ds], genes))
            feats.update({"side_gene_frac": frac, "side_n_spots": n_side})
        key = f"{ds}:{path.stem}"
        sections[key] = df
        rows.append({"key": key, "dataset": ds, "file": path.name, **meta, **feats})
        print(f"done {key}")

scores = pd.DataFrame(rows).set_index("key")
if unknown:
    print("\nLabels not in DEPTH_CLASS (treated as 'ignore'):")
    for ds, l in sorted(unknown):
        print(f"  {ds}: {l!r}")

scores["section"] = pd.to_numeric(scores["section"], errors="coerce")
cols = (["stage", "embryo", "section"] + [f"frac_{c}" for c in DEPTH_VALUE]
        + ["frac_classified", "D_composition", "C_axial_coverage", "L_laterality"])
if COMPUTE_SIDE_HINT:
    cols += ["side_gene_frac", "side_n_spots"]
scores[cols].round(3)

## Section order within each embryo

**HESTA.** For each embryo, `L` is fitted with a valley-shaped profile along the
section number (see the introduction). Columns:
- `L_fit`: fitted score; `fit_rms`: root-mean-square residual;
- `side`: relative side `A` / `B` (before / after the valley). `A`/`B` are
  defined per embryo – which one is left is unknown. A monotone series lies on
  one side only (all `A`). `0` marks the valley section of a V (it can lie on
  either side) and a midline section (`L < MIDLINE_TOL`) at the end of a
  monotone series;
- `L_pair`: score used for pairing (`L_fit` if the fit is good, else `L`);
- `signed_L`: `−L_pair` on side `A`, `+L_pair` on side `B`, `0` for midline
  sections, empty where the side is undetermined.

**MOSTA.** Section numbers do not follow the position; sections are only
ranked by `L`.

In [ ]:
def isotonic(y, increasing=True):
    """Least-squares isotonic regression (pool-adjacent-violators)."""
    y = np.asarray(y, float)
    if not increasing:
        return -isotonic(-y, True)
    vals, wts, lens = [], [], []
    for v in y:
        vals.append(v); wts.append(1.0); lens.append(1)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            w = wts[-2] + wts[-1]
            vals[-2] = (wts[-2] * vals[-2] + wts[-1] * vals[-1]) / w
            wts[-2] = w; lens[-2] += lens[-1]
            vals.pop(); wts.pop(); lens.pop()
    return np.repeat(vals, lens)


def valley_fit(y):
    """Best non-increasing-then-non-decreasing fit. Returns (fit, sse)."""
    y = np.asarray(y, float)
    best, best_sse = None, np.inf
    for k in range(len(y) + 1):
        left = isotonic(y[:k], increasing=False) if k else np.empty(0)
        right = isotonic(y[k:], increasing=True) if k < len(y) else np.empty(0)
        fit = np.concatenate([left, right])
        sse = float(np.sum((y - fit) ** 2))
        if sse < best_sse - 1e-12:
            best, best_sse = fit, sse
    return best, best_sse


def monotone_fit(y):
    """Best monotone (increasing or decreasing) fit. Returns (fit, sse)."""
    y = np.asarray(y, float)
    fits = [isotonic(y, True), isotonic(y, False)]
    sses = [float(np.sum((y - f) ** 2)) for f in fits]
    i = int(np.argmin(sses))
    return fits[i], sses[i]


def null_sse(y):
    """SSE of the valley fit for random orders of y: all orders up to
    EXACT_PERMUTATION_MAX_N sections, else N_PERMUTATIONS random ones."""
    y = np.asarray(y, float)
    if len(y) <= EXACT_PERMUTATION_MAX_N:
        from itertools import permutations
        return np.array([valley_fit(np.array(p))[1] for p in permutations(y)])
    return np.array([valley_fit(RNG.permutation(y))[1] for _ in range(N_PERMUTATIONS)])


scores["L_fit"], scores["fit_rms"], scores["side"] = np.nan, np.nan, ""
embryo_rows, null_dists = [], []
for (ds, stage, embryo), g in scores.groupby(["dataset", "stage", "embryo"]):
    if ds not in ORDERED_DATASETS or len(g) < MIN_SECTIONS_FOR_FIT:
        continue
    g = g.sort_values("section")
    y = g.L_laterality.to_numpy()
    v_fit, v_sse = valley_fit(y)
    m_fit, m_sse = monotone_fit(y)
    is_valley = m_sse - v_sse >= MIN_VALLEY_GAIN
    fit, sse = (v_fit, v_sse) if is_valley else (m_fit, m_sse)
    rms = np.sqrt(sse / len(y))
    bottom = np.isclose(fit, fit.min())
    first, last = np.flatnonzero(bottom)[[0, -1]]
    idx = np.arange(len(y))
    side = np.where(idx < first, "A", np.where(idx > last, "B", "0"))
    if not is_valley:  # monotone: all sections on one side; only a midline section is "0"
        side = np.where(bottom & (fit < MIDLINE_TOL), "0", "A")
    nulls = null_sse(y)
    scores.loc[g.index, "L_fit"] = fit
    scores.loc[g.index, "fit_rms"] = rms
    scores.loc[g.index, "side"] = side
    embryo_rows.append({
        "dataset": ds, "stage": stage, "embryo": embryo, "n_sections": len(y),
        "sections": ",".join(str(int(s)) for s in g.section),
        "L": ",".join(f"{v:.2f}" for v in y),
        "shape": ("flat" if bottom.all() else "valley" if is_valley and 0 < first and last < len(y) - 1
                  else "monotone"),
        "sse_valley": v_sse, "sse_monotone": m_sse, "fit_rms": rms,
        "p_value": float(np.mean(nulls <= v_sse + 1e-12)),
    })
    null_dists.append(nulls)

fit_ok = scores.fit_rms.notna() & (scores.fit_rms <= MAX_FIT_RMS)
scores["L_pair"] = np.where(fit_ok, scores.L_fit, scores.L_laterality)
scores["signed_L"] = np.select(
    [scores.side == "A", scores.side == "B", (scores.side == "0") & (scores.L_pair < MIDLINE_TOL)],
    [-scores.L_pair, scores.L_pair, 0.0], default=np.nan)

embryo_fits = pd.DataFrame(embryo_rows)
embryo_fits.round(3)

### Validation of the score by the HESTA section order

If `L` measures the distance to the midline, the scores of an ordered HESTA
series must be valley-shaped in the section number. Per embryo, the p-value is
the probability that a random order of the same scores fits a valley at least
as well (exact for up to 8 sections). Single short series carry
little evidence (a series of 3 is valley-shaped in 4 of 6 orders, of 4 in 8 of
24, of 5 in 16 of 120), and combining such discrete p-values with Fisher's
method is very conservative. The joint test therefore uses the *total* misfit
over all embryos and draws its null distribution by permuting every embryo's
series independently.

In [ ]:
if len(embryo_fits):
    n_ok = int((embryo_fits.fit_rms <= MAX_FIT_RMS).sum())
    observed = embryo_fits.sse_valley.sum()
    n_draws = 200_000
    total = sum(RNG.choice(d, size=n_draws) for d in null_dists)
    p_joint = (1 + np.sum(total <= observed + 1e-12)) / (1 + n_draws)
    print(f"{n_ok} of {len(embryo_fits)} fitted HESTA embryos have fit RMS <= {MAX_FIT_RMS}")
    print(f"total valley SSE = {observed:.4f};  joint permutation p-value = {p_joint:.2g}")

    k = len(embryo_fits)
    fig, axes = plt.subplots(1, k, figsize=(3.2 * k, 3), squeeze=False)
    side_color = {"A": "#1f77b4", "0": "#2ca02c", "B": "#d62728"}
    for ax, (_, e) in zip(axes.flat, embryo_fits.iterrows()):
        g = scores[(scores.dataset == e.dataset) & (scores.stage == e.stage)
                   & (scores.embryo == e.embryo)].sort_values("section")
        ax.plot(g.section, g.L_fit, "-", color="grey", lw=1)
        for s in ["A", "0", "B"]:
            m = g.side == s
            ax.scatter(g.section[m], g.L_laterality[m], c=side_color[s], label=f"side {s}", zorder=3)
        ax.set_title(f"{e.stage} E{e.embryo}\nRMS={e.fit_rms:.3f}  p={e.p_value:.2f}", fontsize=9)
        ax.set_xlabel("section"); ax.set_ylim(0, 1)
    axes.flat[0].set_ylabel("L"); axes.flat[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "laterality_hesta_profiles.png", dpi=150)
    plt.show()

In [ ]:
order = (scores.sort_values(["dataset", "stage", "embryo", "section"])
               [["dataset", "stage", "embryo", "section", "D_composition", "C_axial_coverage",
                 "L_laterality", "L_fit", "side", "L_pair", "signed_L"]])
order["rank_in_embryo"] = order.groupby(["dataset", "stage", "embryo"]).L_pair.rank(method="min")
order.round(3)

## Visual check

Spots coloured by depth class (red = axial, brown = juxta-axial, orange =
paramedian, blue = lateral, grey = ignored). The score should match the
impression: a continuous red band (spinal cord) for medial sections, a brown
ganglion/somite chain just off the midline, orange kidney/lung further out,
blue limb buds for lateral sections.

In [ ]:
COLORS = {"axial": "#d62728", "juxta_axial": "#8c564b", "paramedian": "#ff7f0e",
          "lateral": "#1f77b4", "ignore": "#d9d9d9"}
keys = list(scores.sort_values(["dataset", "stage", "L_laterality"]).index)
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow), squeeze=False)
for ax, key in zip(axes.flat, keys):
    df = sections[key]
    cls = df.label.str.lower().map(DEPTH_CLASS).fillna("ignore")
    keep = ~df.label.str.lower().isin(NON_TISSUE)
    for c in ["ignore", "paramedian", "juxta_axial", "lateral", "axial"]:
        m = keep & (cls == c)
        ax.scatter(df.x[m], df.y[m], s=0.3, c=COLORS[c], rasterized=True)
    r = scores.loc[key]
    side = f"  side {r.side}" if r.side else ""
    ax.set_title(f"{key}\nD={r.D_composition:.2f}  C={r.C_axial_coverage:.2f}  L={r.L_laterality:.2f}{side}",
                 fontsize=9)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
for ax in axes.flat[len(keys):]:
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=v, label=k) for k, v in COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT_DIR / "laterality_sections.png", dpi=150)
plt.show()

## Pairing HESTA and MOSTA sections

For every HESTA section, the MOSTA sections of the matched stage(s) are ranked
by `|ΔL|`, using `L_pair` for HESTA (smoothed by the series fit) and `L` for
MOSTA. `orientation` says how the pair has to be aligned:
- `as is`: both sections lie at the midline (`L < MIDLINE_TOL`), so the
  mirror ambiguity does not matter;
- `test both`: the side of the MOSTA section is unknown, so the alignment has
  to be run with and without reflecting one section. HESTA sections of one
  embryo with the same `side` must end up with the same choice, which couples
  their alignments.

`best` marks the closest MOSTA section for each HESTA section.

In [ ]:
def norm_stage(s):
    return s.replace("–", "-")

h = scores[scores.dataset == "HESTA"]
m = scores[scores.dataset == "MOSTA"]
pairs = []
for key, r in h.iterrows():
    targets = STAGE_PAIRS.get(norm_stage(r.stage), [])
    cand = m[m.stage.isin(targets)]
    base = {"HESTA": key, "matched_stages": ",".join(targets) or "?",
            "HESTA_embryo": f"{r.stage}_E{r.embryo}", "HESTA_side": r.side or "?",
            "L_HESTA": r.L_pair}
    if cand.empty:
        pairs.append({**base, "MOSTA": None, "L_MOSTA": np.nan, "abs_dL": np.nan,
                      "orientation": "", "note": "no MOSTA file for matched stage(s)"})
        continue
    for mk, mr in cand.iterrows():
        midline = r.L_pair < MIDLINE_TOL and mr.L_laterality < MIDLINE_TOL
        pairs.append({**base, "MOSTA": mk, "L_MOSTA": mr.L_laterality,
                      "abs_dL": abs(r.L_pair - mr.L_laterality),
                      "orientation": "as is" if midline else "test both", "note": ""})
pairs = pd.DataFrame(pairs).sort_values(["HESTA", "abs_dL"])
pairs["best"] = ~pairs.duplicated("HESTA") & pairs.MOSTA.notna()

best = pairs[pairs.best].copy()
print(f"{(best.abs_dL < 0.1).sum()} of {len(best)} HESTA sections have a MOSTA partner with |dL| < 0.1")
best.round(3)

## Save

In [ ]:
scores.to_csv(OUT_DIR / "laterality_scores.csv")
pairs.to_csv(OUT_DIR / "laterality_pairs.csv", index=False)
embryo_fits.to_csv(OUT_DIR / "laterality_hesta_fits.csv", index=False)
print("saved to", OUT_DIR.resolve())